In [1]:
!nvidia-smi

Sat Dec 20 15:45:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -qq -U evaluate rouge_score

In [3]:
import unsloth
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [5]:
max_seq_length = 1024

In [6]:
# Load the base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,   # Define context length
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # Add your token if using a gated model
)

==((====))==  Unsloth 2025.12.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [7]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

Unsloth 2025.12.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [8]:
from datasets import load_dataset
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [9]:
print("Reasoning Dataset Example Row:")
print(reasoning_dataset.shape)
print("\nNon-Reasoning Dataset Example Row (raw):")
print(non_reasoning_dataset.shape)

Reasoning Dataset Example Row:
(19252, 8)

Non-Reasoning Dataset Example Row (raw):
(100000, 3)


In [10]:
reasoning_dataset.column_names

['expected_answer',
 'problem_type',
 'problem_source',
 'generation_model',
 'pass_rate_72b_tir',
 'problem',
 'generated_solution',
 'inference_mode']

In [11]:
def generate_reasoning_conversation(examples):
    problems  = examples["problem"]
    # The 'generated_solution' contains the Chain-of-Thought reasoning
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            # The solution here includes the <think>...</think> block already formatted
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }


In [12]:
# Step 1: Generate conversations
mapped_reasoning = reasoning_dataset.map(
    generate_reasoning_conversation, 
    batched=True, 
    num_proc=1,
    remove_columns=reasoning_dataset.column_names  # Remove original columns
)

# Step 2: Apply chat template
reasoning_formatted_texts = []
for conv in mapped_reasoning["conversations"]:
    formatted = tokenizer.apply_chat_template(conv, tokenize=False)
    reasoning_formatted_texts.append(formatted)

print(f"\nTotal formatted texts: {len(reasoning_formatted_texts)}")

Map (num_proc=1):   0%|          | 0/19252 [00:00<?, ? examples/s]


Total formatted texts: 19252


In [13]:
from unsloth.chat_templates import standardize_sharegpt

# Standardize the ShareGPT format first (if applicable)
standardized_non_reasoning = standardize_sharegpt(non_reasoning_dataset)

# Apply the chat template to each conversation
non_reasoning_formatted_texts = [
    tokenizer.apply_chat_template(conv, tokenize=False)
    for conv in standardized_non_reasoning["conversations"]
]

print(f"\nTotal formatted Non-Reasoning texts: {len(non_reasoning_formatted_texts)}")

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]


Total formatted Non-Reasoning texts: 100000


In [14]:
import pandas as pd
from datasets import Dataset

In [15]:
import pandas as pd

# Assume these are defined already
# reasoning_formatted_texts: list or iterable with 19,252 items
# non_reasoning_formatted_texts: list or iterable with at least 15,000 items

chat_percentage = 0.75  # aim for 75% chat (reasoning) data

reasoning_series = pd.Series(reasoning_formatted_texts)
non_reasoning_series = pd.Series(non_reasoning_formatted_texts)

num_reasoning = 500
num_non_reasoning = 1000

# Ensure we don't oversample from the available data
num_reasoning = min(num_reasoning, len(reasoning_series))
num_non_reasoning = min(num_non_reasoning, len(non_reasoning_series))

# Sample
reasoning_sample = reasoning_series.sample(n=num_reasoning, random_state=42)
non_reasoning_sample = non_reasoning_series.sample(n=num_non_reasoning, random_state=42)

print(f"Using {len(reasoning_sample)} reasoning samples.")
print(f"Sampling {len(non_reasoning_sample)} non-reasoning samples.")


Using 500 reasoning samples.
Sampling 1000 non-reasoning samples.


In [16]:
# # Define desired chat data percentage
# chat_percentage = 0.75 # Aim for 75% chat data
# # Convert to Pandas Series for easier sampling
# reasoning_series = pd.Series(reasoning_formatted_texts)
# non_reasoning_series = pd.Series(non_reasoning_formatted_texts)
# # Sample non-reasoning data based on the desired ratio relative to reasoning data
# # Calculate how many non-reasoning samples we need
# num_non_reasoning_samples = int(len(reasoning_series) * (chat_percentage / (1.0 - chat_percentage)))
# # Ensure we don't request more samples than available
# num_non_reasoning_samples = min(num_non_reasoning_samples, len(non_reasoning_series))

# print(f"Using {len(reasoning_series)} reasoning samples.")
# print(f"Sampling {num_non_reasoning_samples} non-reasoning samples.")

In [17]:
non_reasoning_subset = non_reasoning_series.sample(
    n = len(non_reasoning_sample),
    random_state = 2407, # for reproducibility
)

# Combine the datasets
combined_series = pd.concat([reasoning_series, non_reasoning_subset])
combined_series.name = "text" # The SFTTrainer expects this column name

# Convert back to Hugging Face Dataset and shuffle
combined_dataset = Dataset.from_pandas(pd.DataFrame(combined_series))


combined_dataset = combined_dataset.shuffle(seed = 3407)

# Take the first 1000 rows as a new Dataset
small_dataset = combined_dataset.select(range(1000))

print(f"Small dataset has {len(small_dataset)} rows.")

print(f"\nFinal Combined Dataset size: {len(combined_dataset)}")
#print("Example entry from combined dataset:")
#print(combined_dataset[0]['text'])

Small dataset has 1000 rows.

Final Combined Dataset size: 20252


In [18]:
# Split into 90% train and 10% validation
split_dataset = small_dataset.train_test_split(test_size=0.2, seed=3407)

# Access the train and validation sets
train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]


In [19]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length
    )

In [20]:
train_dataset = train_dataset.map(tokenize_function, batched=True, num_proc=1, remove_columns=["text"])
valid_dataset = valid_dataset.map(tokenize_function, batched=True, num_proc=1, remove_columns=["text"])


Map (num_proc=1):   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/200 [00:00<?, ? examples/s]

In [21]:
import numpy as np
from transformers import EvalPrediction
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")


In [22]:

def preprocess_logits_for_metrics(logits, labels):
    """Returns predicted token IDs (argmax) for metrics calculation"""
    if isinstance(logits, tuple):
        logits = logits[0]  # Unpack if needed
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds
    
    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()
    
    accuracy = (preds_flat == labels_flat).mean()
    
    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    
    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

In [23]:
from trl import SFTTrainer, SFTConfig

In [24]:
sftconfig = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 8, # Effective batch size = 2 * 4 = 8
        warmup_steps = 5,
        max_steps = 30,                 # Short run for demonstration; set to None for full epochs
        # num_train_epochs = 1,         # Alternatively, train for 1 full epoch
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), # Use bf16 if available, else fp16
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",           # Use 8-bit AdamW optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        per_device_eval_batch_size=8,
        seed = 3407,
        dataloader_pin_memory=True, #fast gpu data transfer
        output_dir = "outputs",         # Directory to save checkpoints
        report_to = "none",             # Disable external reporting (like WandB) for this example
        eval_strategy="steps",  # Evaluate during training
        eval_steps=5,                 # Evaluate every 5 steps
        fp16_full_eval = True,
        eval_accumulation_steps=1,
        load_best_model_at_end=True, # Load best model based on evaluation metric
        metric_for_best_model="eval_loss",  # You can also use "eval_loss"
        greater_is_better=False,           # For accuracy, higher is better
        dataset_num_proc=1

    )

In [25]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=sftconfig,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics,
)

In [26]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",       # matches your chat template
    response_part    = "<|im_start|>assistant\n",  # aligns with assistant replies
)


Map (num_proc=6):   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

In [27]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()
print("Training finished.")


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Accuracy,Rouge1,Rouge2,Rougel,Bleu
5,1.149000,1.309932,0.006769,0.809985,0.565552,0.701379,0.553311
10,0.894900,1.216796,0.006764,0.811592,0.571290,0.704328,0.559160
15,0.692400,1.070097,0.006811,0.823474,0.585709,0.716429,0.582059
20,1.021700,1.000076,0.006800,0.830125,0.592627,0.722834,0.588059
25,0.793800,0.937016,0.006795,0.833858,0.596361,0.726942,0.578951
30,0.586000,0.895194,0.006790,0.834047,0.597026,0.727439,0.576294


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training finished.


In [28]:
# print training stats
print(trainer_stats)

TrainOutput(global_step=30, training_loss=0.9296180645624796, metrics={'train_runtime': 992.0063, 'train_samples_per_second': 1.935, 'train_steps_per_second': 0.03, 'total_flos': 5137854627840000.0, 'train_loss': 0.9296180645624796, 'epoch': 2.32})


In [29]:
from transformers import TextStreamer
messages = [
    {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
]

In [30]:
# Format the prompt, explicitly DISABLING thinking mode
text_input_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Crucial for generation
    enable_thinking = False,      # *** Disable thinking ***
)


print("--- Non-Thinking Inference ---")
print("Formatted Input:\n", text_input_no_think)

--- Non-Thinking Inference ---
Formatted Input:
 <|im_start|>user
Solve (x + 2)^2 = 0.<|im_end|>
<|im_start|>assistant
<think>

</think>




In [31]:
# Generate response using parameters suitable for non-thinking/chat
inputs = tokenizer(text_input_no_think, return_tensors = "pt").to("cuda")
streamer_no_think = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    **inputs,
    max_new_tokens = 256,
    temperature = 0.7, # Recommended for chat
    top_p = 0.8,       # Recommended for chat
    top_k = 20,
    streamer = streamer_no_think,
    eos_token_id = tokenizer.eos_token_id # Ensure generation stops properly
)
print("\n-----------------------------")

Решим уравнение $(x + 2)^2 = 0$:

1. Вынесем $x + 2$ в квадрат:
   $$
   (x + 2)^2 = 0
   $$

2. Уравнение квадрата равно нулю, значит:
   $$
   x + 2 = 0
   $$

3. Решим уравнение:
   $$
   x + 2 = 0 \implies x = -2
   $$

**Ответ:** $x = -2$.

Если нужно, можно проверить:
$$
(-2 + 2)^2 = 0^2 = 0
$$
Проверка выполнена. Уравнение верно. Значит, решение правильное. 😊

Если нужно, могу уточнить или решить другое уравнение, если уточнить. 😊

Если нужно, можно также решить уравнение с разными коэффициентами, например, $(x + 2)^2 = 4$, и т.

-----------------------------


## **Thinking Inference:**

In [32]:
# Format the prompt, explicitly ENABLING thinking mode
text_input_think = tokenizer.apply_chat_template(
    messages, # Same user message
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = True,       # *** Enable thinking ***
)

print("--- Thinking Inference ---")
print("Formatted Input:\n", text_input_think)

--- Thinking Inference ---
Formatted Input:
 <|im_start|>user
Solve (x + 2)^2 = 0.<|im_end|>
<|im_start|>assistant



In [33]:
# Generate response using parameters suitable for thinking/reasoning
inputs_think = tokenizer(text_input_think, return_tensors = "pt").to("cuda")
streamer_think = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    **inputs_think,
    max_new_tokens = 1024, # Allow more tokens for reasoning steps
    temperature = 0.6,   # Recommended for reasoning
    top_p = 0.95,        # Recommended for reasoning
    top_k = 20,
    streamer = streamer_think,
    eos_token_id = tokenizer.eos_token_id # Ensure generation stops properly
)
print("\n-----------------------------")

<think>
Okay, so I need to solve the equation (x + 2)^2 = 0. Hmm, let's see. I remember that when you have an equation like this, you can solve for x by isolating it. But wait, the equation is a square, so maybe I should start by taking the square root of both sides to get rid of the square. Let me try that.

So, taking the square root of both sides would give me x + 2 = ±√0. But √0 is 0, right? So that simplifies to x + 2 = 0. Then, to solve for x, I just subtract 2 from both sides. That would be x = 0 - 2, which is x = -2. Wait, so the solution is x equals -2? But let me double-check to make sure I didn't make a mistake.

Let me plug x = -2 back into the original equation. (x + 2)^2 should equal 0. Let's compute that: (-2 + 2)^2 = 0^2 = 0. Yep, that works out. So x = -2 is indeed the correct solution.

But wait, maybe there's another way to approach this. Sometimes, equations like these can have solutions that are not obvious. For example, if the square is zero, then the expression i

In [34]:
# Save LoRA adapters locally
model.save_pretrained("qwen3_0.6b_reasoning_chat_lora")
tokenizer.save_pretrained("qwen3_0.6b_reasoning_chat_lora")

print("LoRA adapters saved locally to 'qwen3_0.6b_reasoning_chat_lora'")

# Optional: Push to Hugging Face Hub
# model.push_to_hub("your_username/qwen3_14b_reasoning_chat_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_username/qwen3_14b_reasoning_chat_lora", token="YOUR_HF_TOKEN")

# To load these adapters later:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "qwen3_14b_reasoning_chat_lora", # Path to saved adapters
#     load_in_4bit = True,
# )

LoRA adapters saved locally to 'qwen3_0.6b_reasoning_chat_lora'
